# Inhibitory Modulation Backbone Workflow

This notebook organizes the inhibitory-modulation analysis into the requested methodological workflow: signed connectome validation, state-space orientation, backbone geometry, E/I decomposition, linear dynamics, propagation, stability and transient-growth diagnostics, inhibitory ablations, motif geometry, and randomized inhibitory null inference.

## tl;dr

Validated on the current 85-node matrix with self-connections retained: 71 nonzero diagonal connections are included in normalization, propagation, stability, ablation, and null analyses. The workflow treats the input matrix as `M[pre, post]`, uses `W = M.T` for the state-space model, and uses the matrix row order as the supplied Hamiltonian/backbone ordering unless `BACKBONE_ORDER` is overridden.

## Context & Methods

The analysis asks how signed inhibitory structure regulates propagation along a directed backbone. Rows in the source matrix are interpreted as presynaptic senders and columns as postsynaptic receivers; the state-space matrix is transposed so columns are senders and rows are receivers.

### Key Assumptions

- `matrices/mij_matrix.csv` is the signed directed connectome in biological orientation, `M_ij = i -> j`.
- Nonzero diagonal entries are retained as self-connections and are tested as their own ablation/motif group.
- The backbone order is the order of labels in the matrix unless `BACKBONE_ORDER` is replaced with a previously identified Hamiltonian ordering.
- Positive signed weights are treated as excitatory effects and negative signed weights as inhibitory effects for this dynamical analysis.
- Matrix exponentials are computed by eigendecomposition, not by Euler stepping.

## Setup

Import dependencies and set analysis parameters. The defaults keep the full workflow runnable on the 85-node matrix while making the null and ablation sections explicit and reproducible.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
    HAS_PLOTS = True
except ImportError:
    HAS_PLOTS = False

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CONNECTIVITY_PATH = PROJECT_ROOT / "matrices" / "mij_matrix.csv"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "01_inhibitory_modulation_backbone_workflow"

ALPHA = 0.85
BETA = 1.0
G_BASELINE = 1.0
G_SWEEP = np.linspace(0.0, 2.0, 21)
TIME_GRID = np.linspace(0.0, 20.0, 121)
EPSILON = 1e-12
DALE_STRONG_THRESHOLD = 0.8
N_NULL = 100
RANDOM_SEED = 11

# Override this with a previously identified Hamiltonian/backbone order if it
# differs from the matrix order.
BACKBONE_ORDER = None

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")
print(f"Output directory: {OUTPUT_DIR}")


## 1. Load And Validate The Signed Directed Connectome

Load the weighted adjacency matrix `M`, confirm it is square and finite, align row and column labels, retain self-connections, and summarize edge signs and density. The biological convention is `M_ij = i -> j`: rows are presynaptic senders and columns are postsynaptic receivers.

In [ ]:
raw_M = pd.read_csv(CONNECTIVITY_PATH, index_col=0)
raw_M.index = raw_M.index.astype(str)
raw_M.columns = raw_M.columns.astype(str)
raw_M = raw_M.apply(pd.to_numeric, errors="coerce")

if raw_M.shape[0] != raw_M.shape[1]:
    raise ValueError(f"Connectivity matrix must be square, got {raw_M.shape}.")
if set(raw_M.index) != set(raw_M.columns):
    raise ValueError("Row and column labels do not contain the same neuron set.")
if not np.isfinite(raw_M.to_numpy(dtype=float)).all():
    raise ValueError("Connectivity matrix contains non-finite values.")

M = raw_M.reindex(index=raw_M.index, columns=raw_M.index).fillna(0.0).copy()

M_values = M.to_numpy(dtype=float)
diagonal_values = np.diag(M_values)
off_diagonal_mask = ~np.eye(M.shape[0], dtype=bool)
edge_summary = pd.Series(
    {
        "nodes": M.shape[0],
        "entries": M_values.size,
        "positive_edges": int(np.count_nonzero(M_values > 0)),
        "negative_edges": int(np.count_nonzero(M_values < 0)),
        "zero_entries": int(np.count_nonzero(M_values == 0)),
        "self_connections_included": int(np.count_nonzero(diagonal_values)),
        "positive_self_connections": int(np.count_nonzero(diagonal_values > 0)),
        "negative_self_connections": int(np.count_nonzero(diagonal_values < 0)),
        "density_including_self": float(np.count_nonzero(M_values) / M_values.size),
        "density_without_self": float(np.count_nonzero(M_values[off_diagonal_mask]) / off_diagonal_mask.sum()),
    }
).to_frame("value")

display(edge_summary)
display(M.iloc[:5, :5])

## 2. Convert To State-Space Orientation

The dynamical system is `dx/dt = A x`, where `A_ij` is the effect of node `j` on node `i`. Because the source connectome uses rows as senders, the state-space connectivity is `W = M.T`, with rows as receivers and columns as senders.


In [ ]:
W = M.T.copy()

orientation_check = pd.DataFrame(
    {
        "matrix": ["M", "W"],
        "row_meaning": ["presynaptic sender", "postsynaptic receiver"],
        "column_meaning": ["postsynaptic receiver", "presynaptic sender"],
        "shape": [str(M.shape), str(W.shape)],
    }
)
display(orientation_check)


## 3. Define The Directed Backbone

Supply the Hamiltonian/backbone ordering `P = (v1, v2, ..., vn)` and assign each neuron a coordinate `p(v_k) = k`. This coordinate is the reference geometry for propagation and inhibition.


In [ ]:
if BACKBONE_ORDER is None:
    backbone_order = list(M.index)
else:
    missing = set(BACKBONE_ORDER).difference(M.index)
    extra = set(M.index).difference(BACKBONE_ORDER)
    if missing or extra:
        raise ValueError(f"Backbone order mismatch. Missing in matrix: {missing}; omitted nodes: {extra}")
    backbone_order = list(BACKBONE_ORDER)

backbone_position = pd.Series(np.arange(len(backbone_order)), index=backbone_order, name="backbone_position")
start_node = backbone_order[0]
terminal_node = backbone_order[-1]

display(backbone_position.head(10).to_frame())
print(f"Backbone start: {start_node}")
print(f"Backbone terminal: {terminal_node}")


## 4. Normalize Connection Strengths

Normalize the state-space weights by the 95th percentile of nonzero absolute weights. This preserves signs and relative magnitudes while reducing domination by very large weights.


In [ ]:
nonzero_abs_weights = np.abs(W.to_numpy(dtype=float)[W.to_numpy(dtype=float) != 0])
robust_scale = float(np.quantile(nonzero_abs_weights, 0.95)) if len(nonzero_abs_weights) else 1.0
if robust_scale == 0:
    robust_scale = 1.0

W_hat = W / robust_scale
normalization_summary = pd.Series(
    {
        "nonzero_abs_q95": robust_scale,
        "max_abs_before": float(nonzero_abs_weights.max()) if len(nonzero_abs_weights) else 0.0,
        "max_abs_after": float(np.abs(W_hat.to_numpy(dtype=float)).max()),
    }
).to_frame("value")
display(normalization_summary)


## 5. Separate Excitatory And Inhibitory Connectivity

Decompose the signed normalized matrix into positive excitatory effects `W_E = max(W_hat, 0)` and inhibitory magnitudes `W_I = max(-W_hat, 0)`, so `W_hat = W_E - W_I`.


In [ ]:
W_E = W_hat.clip(lower=0.0)
W_I = (-W_hat).clip(lower=0.0)
reconstruction_error = float(np.abs((W_E - W_I - W_hat).to_numpy(dtype=float)).max())

ei_summary = pd.Series(
    {
        "excitatory_nonzero": int(np.count_nonzero(W_E.to_numpy(dtype=float))),
        "inhibitory_nonzero": int(np.count_nonzero(W_I.to_numpy(dtype=float))),
        "reconstruction_max_abs_error": reconstruction_error,
    }
).to_frame("value")
display(ei_summary)


## 6. Construct The Linear Excitatory-Inhibitory Dynamical System

Use `A(g) = -alpha I + beta * (W_E - g W_I)` and `dx/dt = A(g)x + u(t)`. Increasing `g` progressively strengthens inhibitory connectivity.


In [ ]:
def build_A(g, W_E_matrix=W_E, W_I_matrix=W_I, alpha=ALPHA, beta=BETA):
    values = beta * (W_E_matrix.to_numpy(dtype=float) - float(g) * W_I_matrix.to_numpy(dtype=float))
    values = values - alpha * np.eye(values.shape[0])
    return pd.DataFrame(values, index=W_E_matrix.index, columns=W_E_matrix.columns)

A_baseline = build_A(G_BASELINE)
display(pd.Series({"alpha": ALPHA, "beta": BETA, "baseline_g": G_BASELINE}).to_frame("value"))
display(A_baseline.iloc[:5, :5])


## 7. Perform A Dale's-Law Diagnostic

Inspect each neuron's outgoing edges in the original biological orientation. Neurons with mostly positive outgoing weights are labeled mostly excitatory, mostly negative outgoing weights mostly inhibitory, and the remainder mixed-sign.


In [ ]:
dale_rows = []
for neuron, outgoing in M.iterrows():
    nonzero = outgoing[outgoing != 0]
    positive_fraction = float((nonzero > 0).mean()) if len(nonzero) else np.nan
    negative_fraction = float((nonzero < 0).mean()) if len(nonzero) else np.nan
    if len(nonzero) == 0:
        classification = "no outgoing edges"
    elif positive_fraction >= DALE_STRONG_THRESHOLD:
        classification = "mostly excitatory"
    elif negative_fraction >= DALE_STRONG_THRESHOLD:
        classification = "mostly inhibitory"
    else:
        classification = "mixed-sign"
    dale_rows.append(
        {
            "neuron": neuron,
            "outgoing_edges": int(len(nonzero)),
            "positive_fraction": positive_fraction,
            "negative_fraction": negative_fraction,
            "classification": classification,
        }
    )

dale_diagnostic = pd.DataFrame(dale_rows).sort_values(["classification", "outgoing_edges"], ascending=[True, False])
display(dale_diagnostic["classification"].value_counts().to_frame("neuron_count"))
display(dale_diagnostic.head(15))


## 8. Simulate Impulse Propagation

Apply a unit perturbation to the first backbone neuron, `x(0) = e_v1`, and evaluate `x(t) = exp(A(g)t)x(0)` on the time grid.


In [ ]:
def eigensystem(A_values):
    eigenvalues, eigenvectors = np.linalg.eig(A_values)
    inverse_vectors = np.linalg.pinv(eigenvectors)
    condition_number = float(np.linalg.cond(eigenvectors))
    return eigenvalues, eigenvectors, inverse_vectors, condition_number


def propagate_impulse(A_frame, source_node, times):
    A_values = A_frame.to_numpy(dtype=float)
    eigenvalues, eigenvectors, inverse_vectors, condition_number = eigensystem(A_values)
    x0 = np.zeros(A_values.shape[0], dtype=float)
    x0[A_frame.index.get_loc(source_node)] = 1.0
    coefficients = inverse_vectors @ x0
    states = []
    for t in times:
        state = eigenvectors @ (np.exp(eigenvalues * float(t)) * coefficients)
        states.append(np.real_if_close(state, tol=1000).real)
    return pd.DataFrame(states, index=times, columns=A_frame.index), condition_number


baseline_states, baseline_eigen_condition = propagate_impulse(A_baseline, start_node, TIME_GRID)
display(baseline_states.iloc[:5, :8])
print(f"Eigenvector condition number for baseline A: {baseline_eigen_condition:.3g}")


## 9. Measure Activity Along The Backbone

For every backbone neuron, measure peak absolute activity, integrated absolute activity, and time to peak. These summarize attenuation, persistence, and propagation timing.


In [ ]:
def backbone_activity_metrics(states, order, label):
    ordered = states.loc[:, order]
    abs_values = ordered.abs()
    peak_activity = abs_values.max(axis=0)
    auc = pd.Series(np.trapz(abs_values.to_numpy(dtype=float), x=states.index.to_numpy(dtype=float), axis=0), index=order)
    time_to_peak = abs_values.idxmax(axis=0)
    return pd.DataFrame(
        {
            "scenario": label,
            "neuron": order,
            "backbone_position": np.arange(len(order)),
            "peak_abs_activity": peak_activity.reindex(order).to_numpy(dtype=float),
            "auc_abs_activity": auc.reindex(order).to_numpy(dtype=float),
            "time_to_peak": time_to_peak.reindex(order).to_numpy(dtype=float),
        }
    )


baseline_metrics = backbone_activity_metrics(baseline_states, backbone_order, f"g={G_BASELINE:g}")
display(baseline_metrics.head(12))
display(baseline_metrics.tail(8))


## 10. Calculate An Inhibition Index

Compare peak activity with inhibition against the `g = 0` system using `I_k(g) = 1 - P_k(g) / (P_k(0) + epsilon)`. Positive values indicate suppression; negative values indicate indirect amplification.


In [ ]:
A_no_inhibition = build_A(0.0)
no_inhibition_states, _ = propagate_impulse(A_no_inhibition, start_node, TIME_GRID)
no_inhibition_metrics = backbone_activity_metrics(no_inhibition_states, backbone_order, "g=0")

inhibition_index = baseline_metrics[["neuron", "backbone_position", "peak_abs_activity"]].rename(
    columns={"peak_abs_activity": "peak_with_inhibition"}
).merge(
    no_inhibition_metrics[["neuron", "peak_abs_activity"]].rename(columns={"peak_abs_activity": "peak_without_inhibition"}),
    on="neuron",
    how="left",
)
inhibition_index["inhibition_index"] = 1.0 - (
    inhibition_index["peak_with_inhibition"] / (inhibition_index["peak_without_inhibition"] + EPSILON)
)

display(inhibition_index.head(12))
display(inhibition_index.tail(8))


## 11. Sweep Inhibitory Strength

Vary `g` from 0 to 2 and measure terminal response, terminal AUC, time to peak, total backbone activity, and dynamical stability. This creates an inhibition-response curve instead of relying on one arbitrary inhibitory strength.


In [ ]:
def stability_metrics(A_frame):
    eigenvalues = np.linalg.eigvals(A_frame.to_numpy(dtype=float))
    spectral_abscissa = float(np.max(eigenvalues.real))
    return spectral_abscissa, bool(spectral_abscissa < 0)


sweep_rows = []
metrics_by_g = {}
states_by_g = {}

for g in G_SWEEP:
    A_g = build_A(float(g))
    states_g, _ = propagate_impulse(A_g, start_node, TIME_GRID)
    metrics_g = backbone_activity_metrics(states_g, backbone_order, f"g={g:.2f}")
    spectral_abscissa, stable = stability_metrics(A_g)
    terminal_row = metrics_g.loc[metrics_g["neuron"] == terminal_node].iloc[0]
    sweep_rows.append(
        {
            "g": float(g),
            "terminal_peak_abs_activity": float(terminal_row["peak_abs_activity"]),
            "terminal_auc_abs_activity": float(terminal_row["auc_abs_activity"]),
            "terminal_time_to_peak": float(terminal_row["time_to_peak"]),
            "total_backbone_auc": float(metrics_g["auc_abs_activity"].sum()),
            "spectral_abscissa": spectral_abscissa,
            "stable_continuous": stable,
        }
    )
    metrics_by_g[float(g)] = metrics_g
    states_by_g[float(g)] = states_g

sweep_summary = pd.DataFrame(sweep_rows)
display(sweep_summary)

if HAS_PLOTS:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4), dpi=140)
    axes[0].plot(sweep_summary["g"], sweep_summary["terminal_peak_abs_activity"], marker="o")
    axes[0].set_xlabel("inhibitory scale g")
    axes[0].set_ylabel("terminal peak activity")
    axes[0].set_title("Terminal propagation")
    axes[1].plot(sweep_summary["g"], sweep_summary["spectral_abscissa"], marker="o", color="tab:red")
    axes[1].axhline(0, color="black", linewidth=1)
    axes[1].set_xlabel("inhibitory scale g")
    axes[1].set_ylabel("spectral abscissa")
    axes[1].set_title("Linear stability")
    plt.tight_layout()
    fig.savefig(OUTPUT_DIR / "inhibitory_strength_sweep.png", bbox_inches="tight")
    plt.show()


## 12. Analyze Linear Stability

Calculate eigenvalues of `A(g)` and track the spectral abscissa `Lambda(g) = max Re(lambda_i(A(g)))`. Negative values indicate asymptotic stability.


In [ ]:
eigenvalue_rows = []
for g in G_SWEEP:
    eigenvalues = np.linalg.eigvals(build_A(float(g)).to_numpy(dtype=float))
    for eigenvalue in eigenvalues:
        eigenvalue_rows.append(
            {
                "g": float(g),
                "real": float(eigenvalue.real),
                "imag": float(eigenvalue.imag),
                "abs": float(abs(eigenvalue)),
            }
        )

eigenvalues_by_g = pd.DataFrame(eigenvalue_rows)
spectral_abscissa_by_g = eigenvalues_by_g.groupby("g", as_index=False)["real"].max().rename(
    columns={"real": "spectral_abscissa"}
)
display(spectral_abscissa_by_g)


## 13. Analyze Transient Amplification

Directed non-normal networks can transiently amplify activity even when all eigenvalues predict eventual decay. Estimate `G(t) = ||exp(A(g)t)||_2` on the time grid and record `G_max`.


In [ ]:
def matrix_exponential_norms(A_frame, times):
    A_values = A_frame.to_numpy(dtype=float)
    eigenvalues, eigenvectors, inverse_vectors, condition_number = eigensystem(A_values)
    rows = []
    for t in times:
        exp_A_t = eigenvectors @ (np.exp(eigenvalues * float(t))[:, None] * inverse_vectors)
        exp_A_t = np.real_if_close(exp_A_t, tol=1000).real
        rows.append({"time": float(t), "G_t": float(np.linalg.norm(exp_A_t, ord=2))})
    return pd.DataFrame(rows), condition_number


transient_rows = []
for g in [0.0, G_BASELINE, 2.0]:
    transient_curve, condition_number = matrix_exponential_norms(build_A(g), TIME_GRID)
    peak_row = transient_curve.loc[transient_curve["G_t"].idxmax()]
    transient_rows.append(
        {
            "g": float(g),
            "G_max": float(peak_row["G_t"]),
            "time_of_G_max": float(peak_row["time"]),
            "eigenvector_condition_number": condition_number,
        }
    )

transient_summary = pd.DataFrame(transient_rows)
display(transient_summary)


## 14. Calculate The Numerical Abscissa

Evaluate `omega(A) = lambda_max((A + A.T) / 2)`, an additional measure of short-term transient growth capacity.


In [ ]:
numerical_abscissa_rows = []
for g in G_SWEEP:
    A_values = build_A(float(g)).to_numpy(dtype=float)
    symmetric_part = (A_values + A_values.T) / 2.0
    omega = float(np.linalg.eigvalsh(symmetric_part).max())
    numerical_abscissa_rows.append({"g": float(g), "numerical_abscissa": omega})

numerical_abscissa = pd.DataFrame(numerical_abscissa_rows)
display(numerical_abscissa)


## 15. Perform Inhibitory-Neuron Ablations

Remove the outgoing inhibitory connections of one source neuron at a time, recompute terminal backbone propagation, and rank neurons by `Delta_i = F(W_I^(-i)) - F(W_I)`, where `F` is terminal peak activity.


In [ ]:
def terminal_peak_for_weights(W_E_candidate=W_E, W_I_candidate=W_I, g=G_BASELINE):
    A_candidate = build_A(g, W_E_matrix=W_E_candidate, W_I_matrix=W_I_candidate)
    states_candidate, _ = propagate_impulse(A_candidate, start_node, TIME_GRID)
    metrics_candidate = backbone_activity_metrics(states_candidate, backbone_order, f"g={g:.2f}")
    return float(metrics_candidate.loc[metrics_candidate["neuron"] == terminal_node, "peak_abs_activity"].iloc[0])


def terminal_peak_for_WI(W_I_candidate, g=G_BASELINE):
    return terminal_peak_for_weights(W_E, W_I_candidate, g=g)


baseline_terminal_peak = terminal_peak_for_WI(W_I)
inhibitory_source_strength = W_I.sum(axis=0)
candidate_inhibitory_sources = inhibitory_source_strength[inhibitory_source_strength > 0].sort_values(ascending=False)

neuron_ablation_rows = []
for neuron, outgoing_strength in candidate_inhibitory_sources.items():
    ablated_W_I = W_I.copy()
    ablated_W_I.loc[:, neuron] = 0.0
    ablated_terminal_peak = terminal_peak_for_WI(ablated_W_I)
    neuron_ablation_rows.append(
        {
            "neuron": neuron,
            "outgoing_inhibitory_strength": float(outgoing_strength),
            "outgoing_inhibitory_edges": int(np.count_nonzero(W_I[neuron].to_numpy(dtype=float))),
            "baseline_terminal_peak": baseline_terminal_peak,
            "ablated_terminal_peak": ablated_terminal_peak,
            "delta_terminal_peak": ablated_terminal_peak - baseline_terminal_peak,
        }
    )

inhibitory_neuron_ablation = pd.DataFrame(neuron_ablation_rows).sort_values(
    "delta_terminal_peak", ascending=False
)
display(inhibitory_neuron_ablation.head(20))

## 16. Perform Inhibitory-Edge And Self-Connection Ablations

Remove individual inhibitory projections `j -| i`, recompute terminal transmission, and rank edges by `Delta_ij = F(W_I^(-ij)) - F(W_I)`. Self-connections are also tested as their own ablation group by zeroing the diagonal in both excitatory and inhibitory components.

In [ ]:
edge_ablation_rows = []
inhibitory_edge_locations = np.argwhere(W_I.to_numpy(dtype=float) > 0)
for receiver_index, sender_index in inhibitory_edge_locations:
    receiver = W_I.index[receiver_index]
    sender = W_I.columns[sender_index]
    ablated_W_I = W_I.copy()
    ablated_W_I.iat[receiver_index, sender_index] = 0.0
    ablated_terminal_peak = terminal_peak_for_WI(ablated_W_I)
    edge_ablation_rows.append(
        {
            "sender": sender,
            "receiver": receiver,
            "inhibitory_weight": float(W_I.iat[receiver_index, sender_index]),
            "is_self_connection": bool(sender == receiver),
            "baseline_terminal_peak": baseline_terminal_peak,
            "ablated_terminal_peak": ablated_terminal_peak,
            "delta_terminal_peak": ablated_terminal_peak - baseline_terminal_peak,
        }
    )

inhibitory_edge_ablation = pd.DataFrame(edge_ablation_rows).sort_values(
    "delta_terminal_peak", ascending=False
)

self_ablated_W_E = W_E.copy()
self_ablated_W_I = W_I.copy()
for idx in range(len(self_ablated_W_E)):
    self_ablated_W_E.iat[idx, idx] = 0.0
    self_ablated_W_I.iat[idx, idx] = 0.0
self_ablated_terminal_peak = terminal_peak_for_weights(self_ablated_W_E, self_ablated_W_I)
self_connection_ablation = pd.DataFrame(
    [
        {
            "group": "self_connections",
            "self_connection_edges": int(np.count_nonzero(np.diag(W_hat.to_numpy(dtype=float)))),
            "positive_self_connections": int(np.count_nonzero(np.diag(W_hat.to_numpy(dtype=float)) > 0)),
            "negative_self_connections": int(np.count_nonzero(np.diag(W_hat.to_numpy(dtype=float)) < 0)),
            "baseline_terminal_peak": baseline_terminal_peak,
            "ablated_terminal_peak": self_ablated_terminal_peak,
            "delta_terminal_peak": self_ablated_terminal_peak - baseline_terminal_peak,
        }
    ]
)

print("Top individual inhibitory-edge ablations")
display(inhibitory_edge_ablation.head(20))
print("Self-connection group ablation")
display(self_connection_ablation)

## 17. Map Inhibition Onto Backbone Geometry

For every inhibitory connection `i -> j`, calculate `d_ij = p(j) - p(i)`. Positive distances are feedforward inhibition, negative distances are feedback inhibition, and large absolute distances are long-range inhibition.


In [ ]:
motif_rows = []
for receiver_index, sender_index in inhibitory_edge_locations:
    receiver = W_I.index[receiver_index]
    sender = W_I.columns[sender_index]
    distance = int(backbone_position.loc[receiver] - backbone_position.loc[sender])
    if distance > 0:
        direction_class = "feedforward inhibition"
    elif distance < 0:
        direction_class = "feedback inhibition"
    else:
        direction_class = "same-coordinate/self"
    motif_rows.append(
        {
            "sender": sender,
            "receiver": receiver,
            "inhibitory_weight": float(W_I.iat[receiver_index, sender_index]),
            "backbone_distance": distance,
            "abs_backbone_distance": abs(distance),
            "direction_class": direction_class,
            "long_range": abs(distance) >= np.quantile(np.abs(backbone_position.to_numpy()[:, None] - backbone_position.to_numpy()[None, :]).ravel(), 0.75),
        }
    )

inhibitory_geometry = pd.DataFrame(motif_rows)
display(inhibitory_geometry.head(15))
display(inhibitory_geometry["direction_class"].value_counts().to_frame("edge_count"))


## 18. Compare Inhibitory Motif Classes

Compare feedforward and feedback inhibitory edges by abundance, strength, backbone distance, and dynamical ablation effect.


In [ ]:
edge_effects = inhibitory_edge_ablation[["sender", "receiver", "delta_terminal_peak"]]
geometry_with_effects = inhibitory_geometry.merge(edge_effects, on=["sender", "receiver"], how="left")

motif_class_comparison = (
    geometry_with_effects.groupby("direction_class", as_index=False)
    .agg(
        edge_count=("sender", "size"),
        median_inhibitory_weight=("inhibitory_weight", "median"),
        mean_inhibitory_weight=("inhibitory_weight", "mean"),
        median_backbone_distance=("backbone_distance", "median"),
        median_abs_backbone_distance=("abs_backbone_distance", "median"),
        long_range_rate=("long_range", "mean"),
        median_delta_terminal_peak=("delta_terminal_peak", "median"),
        mean_delta_terminal_peak=("delta_terminal_peak", "mean"),
    )
    .sort_values("edge_count", ascending=False)
)
display(motif_class_comparison)


## 19. Construct Randomized Inhibitory Null Networks

Randomize inhibitory targets while preserving each inhibitory source's out-degree and outgoing inhibitory weight distribution. Self-targets remain eligible so diagonal inhibitory placement is included in the null model rather than silently excluded. Recompute the terminal propagation statistic for each randomized network.

In [ ]:
def randomize_inhibitory_targets(W_I_observed, rng):
    randomized = pd.DataFrame(0.0, index=W_I_observed.index, columns=W_I_observed.columns)
    receivers = np.array(W_I_observed.index)
    for sender in W_I_observed.columns:
        weights = W_I_observed[sender].to_numpy(dtype=float)
        nonzero_weights = weights[weights > 0]
        if len(nonzero_weights) == 0:
            continue
        sample_size = min(len(nonzero_weights), len(receivers))
        sampled_receivers = rng.choice(receivers, size=sample_size, replace=False)
        sampled_weights = rng.permutation(nonzero_weights)[:sample_size]
        randomized.loc[sampled_receivers, sender] = sampled_weights
    return randomized


rng = np.random.default_rng(RANDOM_SEED)
null_rows = []
for draw in range(N_NULL):
    randomized_W_I = randomize_inhibitory_targets(W_I, rng)
    null_rows.append(
        {
            "draw": draw,
            "terminal_peak": terminal_peak_for_WI(randomized_W_I),
            "randomized_self_inhibitory_edges": int(np.count_nonzero(np.diag(randomized_W_I.to_numpy(dtype=float)))),
        }
    )

inhibitory_null = pd.DataFrame(null_rows)
display(inhibitory_null.describe())

## 20. Quantify Whether Observed Inhibition Is Unusually Organized

Compare observed terminal propagation with the randomized inhibitory null distribution using `Z = (F_obs - mean(F_null)) / sd(F_null)`.


In [ ]:
null_mean = float(inhibitory_null["terminal_peak"].mean())
null_sd = float(inhibitory_null["terminal_peak"].std(ddof=1))
organization_z = (baseline_terminal_peak - null_mean) / null_sd if null_sd > 0 else np.nan
organization_summary = pd.Series(
    {
        "observed_terminal_peak": baseline_terminal_peak,
        "null_mean_terminal_peak": null_mean,
        "null_sd_terminal_peak": null_sd,
        "z_score": organization_z,
        "empirical_p_null_ge_observed": float((np.sum(inhibitory_null["terminal_peak"] >= baseline_terminal_peak) + 1) / (len(inhibitory_null) + 1)),
        "empirical_p_null_le_observed": float((np.sum(inhibitory_null["terminal_peak"] <= baseline_terminal_peak) + 1) / (len(inhibitory_null) + 1)),
    }
).to_frame("value")
display(organization_summary)


## Save Tables

Save compact workflow outputs so the analysis can be inspected outside the notebook.


In [ ]:
edge_summary.to_csv(OUTPUT_DIR / "connectome_edge_summary.csv")
normalization_summary.to_csv(OUTPUT_DIR / "normalization_summary.csv")
dale_diagnostic.to_csv(OUTPUT_DIR / "dale_diagnostic.csv", index=False)
baseline_metrics.to_csv(OUTPUT_DIR / "baseline_backbone_activity_metrics.csv", index=False)
inhibition_index.to_csv(OUTPUT_DIR / "inhibition_index_by_backbone_node.csv", index=False)
sweep_summary.to_csv(OUTPUT_DIR / "inhibitory_strength_sweep.csv", index=False)
spectral_abscissa_by_g.to_csv(OUTPUT_DIR / "spectral_abscissa_by_g.csv", index=False)
transient_summary.to_csv(OUTPUT_DIR / "transient_amplification_summary.csv", index=False)
numerical_abscissa.to_csv(OUTPUT_DIR / "numerical_abscissa_by_g.csv", index=False)
inhibitory_neuron_ablation.to_csv(OUTPUT_DIR / "inhibitory_neuron_ablation.csv", index=False)
inhibitory_edge_ablation.to_csv(OUTPUT_DIR / "inhibitory_edge_ablation.csv", index=False)
self_connection_ablation.to_csv(OUTPUT_DIR / "self_connection_ablation.csv", index=False)
inhibitory_geometry.to_csv(OUTPUT_DIR / "inhibitory_backbone_geometry.csv", index=False)
motif_class_comparison.to_csv(OUTPUT_DIR / "inhibitory_motif_class_comparison.csv", index=False)
inhibitory_null.to_csv(OUTPUT_DIR / "randomized_inhibitory_null.csv", index=False)
organization_summary.to_csv(OUTPUT_DIR / "inhibitory_organization_z_score.csv")

print(f"Saved workflow tables to {OUTPUT_DIR.resolve()}")

## Takeaways

- The notebook follows the requested methodological order from connectome validation through null-model inference with self-connections retained.
- The main dynamical readout is terminal backbone transmission, compared across inhibitory strength, inhibitory-neuron ablations, inhibitory-edge ablations, the explicit self-connection ablation group, inhibitory motif geometry, and randomized inhibitory target placement.
- Interpret high-ranking inhibitory neurons, inhibitory edges, and the self-connection group as candidate gates; interpret the null-model z-score as evidence about whether inhibitory placement along the backbone is unusually organized.